In [9]:
!pip install -q PyMuPDF pydantic deep-translator

import os
import json
import fitz
from typing import List
from pydantic import BaseModel
from google.colab import files

class TextSpanV1(BaseModel):
    text: str
    size: float
    bbox: List[float]

class TextLineV1(BaseModel):
    spans: List[TextSpanV1]
    bbox: List[float]

class TextBlockV1(BaseModel):
    block_id: int
    type: int
    lines: List[TextLineV1]
    bbox: List[float]

class PageV1(BaseModel):
    page_num: int
    width: float
    height: float
    blocks: List[TextBlockV1]

class DocumentV1(BaseModel):
    file_name: str
    total_pages: int
    pages: List[PageV1]

class NormalizeStage:
    def run(self, pdf_path: str, output_json_path: str):
        doc = fitz.open(pdf_path)
        pages_data = []

        for page_num in range(len(doc)):
            page = doc[page_num]
            page_dict = page.get_text("dict")
            blocks_data = []

            for block in page_dict.get('blocks', []):
                if block.get('type') != 0:
                    continue

                lines_data = []
                for line in block.get('lines', []):
                    spans_data = []
                    for span in line.get('spans', []):
                        spans_data.append(TextSpanV1(
                            text=span['text'],
                            size=span['size'],
                            bbox=span['bbox']
                        ))
                    if spans_data:
                        lines_data.append(TextLineV1(
                            spans=spans_data,
                            bbox=line['bbox']
                        ))

                if lines_data:
                    blocks_data.append(TextBlockV1(
                        block_id=block['number'],
                        type=0,
                        lines=lines_data,
                        bbox=block['bbox']
                    ))

            pages_data.append(PageV1(
                page_num=page_num,
                width=page.rect.width,
                height=page.rect.height,
                blocks=blocks_data
            ))

        document = DocumentV1(
            file_name=os.path.basename(pdf_path),
            total_pages=len(doc),
            pages=pages_data
        )

        os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
        with open(output_json_path, 'w', encoding='utf-8') as f:
            # Da sua loi Pydantic V2 tai day
            doc_dict = document.model_dump()
            f.write(json.dumps(doc_dict, indent=2, ensure_ascii=False))

        doc.close()
        print(f"[Normalize Stage] Saved: {output_json_path}")

uploaded = files.upload()
input_pdf = list(uploaded.keys())[0]

workspace_dir = "workspace"
document_v1_path = os.path.join(workspace_dir, "document.v1.json")

normalize_stage = NormalizeStage()
normalize_stage.run(input_pdf, document_v1_path)

Saving 1-s2.0-S1877050925015984-main.pdf to 1-s2.0-S1877050925015984-main.pdf
[Normalize Stage] Saved: workspace/document.v1.json


In [10]:
# Cài đặt các thư viện cần thiết để chạy Local Model
!pip install -q transformers accelerate bitsandbytes tqdm pydantic

import os
import json
import re
import torch
from tqdm.auto import tqdm
from pydantic import BaseModel
from typing import List
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class TranslatedBlockV1(BaseModel):
    block_id: int
    original_text: str
    translated_text: str

class PageTranslationPayload(BaseModel):
    page_num: int
    translated_blocks: List[TranslatedBlockV1]

class TranslationManifest(BaseModel):
    file_name: str
    pages: List[PageTranslationPayload]

class TranslationStage:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        print(f"[Translation Stage] Đang tải mô hình {model_id} vào GPU...")

        # Thiết lập lượng tử hóa 4-bit để tối ưu hóa VRAM (ép xuống < 6GB)
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            device_map="auto" # Tự động phân bổ vào GPU
        )
        print("[Translation Stage] Mô hình đã sẵn sàng!")

    def is_protected_content(self, text: str, block_lines: int) -> bool:
        if len(text) < 4 or re.match(r'^[\W_0-9]+$', text):
            return True

        patterns = [
            r'(\$.+?\$|\\\[.+?\\\]|\\\(.+?\\\))',
            r'(def |class |import |return |if |else:|for |while |#include)',
            r'([=\+\-\*\/\%\^\&\|\>\<\$]{2,}|[a-zA-Z_]+\(.*\)|\{.*\})',
            r'(https?://\S+|www\.\S+)',
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            r'^[A-Z0-9_]+$'
        ]

        for p in patterns:
            matches = re.findall(p, text)
            if sum(len(m) for m in matches) > len(text) * 0.25:
                return True

        if block_lines > 1 and (len(text) / block_lines) < 15:
            if not re.search(r'[.!?]$', text):
                return True

        return False

    def translate_block(self, text: str) -> str:
        # Nếu đoạn text quá dài, có thể cắt nhỏ (mặc dù Local Model có context window lớn)
        if len(text) > 2000:
            parts = [text[i:i+2000] for i in range(0, len(text), 2000)]
            return " ".join([self._generate_translation(part) for part in parts])
        return self._generate_translation(text)

    def _generate_translation(self, text: str) -> str:
        prompt = (
            "Dịch đoạn văn bản tiếng Anh sau sang tiếng Việt một cách chính xác. "
            "Giữ nguyên văn phong học thuật, chuyên nghiệp. "
            "CHỈ trả về nội dung đã dịch, không giải thích, không dùng định dạng markdown.\n\n"
            f"Văn bản gốc: {text}"
        )

        messages = [
            {"role": "system", "content": "Bạn là một chuyên gia dịch thuật tài liệu khoa học chuyên nghiệp."},
            {"role": "user", "content": prompt}
        ]

        text_input = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text_input], return_tensors="pt").to(self.model.device)

        # Tắt gradient để tiết kiệm bộ nhớ khi suy luận (inference)
        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.3, # Nhiệt độ thấp để dịch thuật chính xác, bớt bịa chữ
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response.strip()

    def run(self, input_json_path: str, output_manifest_path: str):
        with open(input_json_path, 'r', encoding='utf-8') as f:
            document_data = json.load(f)

        file_name = document_data.get('file_name', 'unknown.pdf')
        pages_data = document_data.get('pages', [])

        manifest_pages = []

        # Xử lý tuần tự do Local GPU chạy batching thủ công phức tạp hơn
        print(f"[Translation Stage] Bắt đầu dịch {len(pages_data)} trang bằng mô hình cục bộ...")

        for page in tqdm(pages_data, desc="Tiến trình dịch tài liệu", unit="trang"):
            translated_blocks = []

            for block in page.get('blocks', []):
                original_text = ""
                for line in block.get('lines', []):
                    for span in line.get('spans', []):
                        original_text += span.get('text', '') + " "

                original_text = re.sub(r'-\s*\n', '', original_text)
                original_text = re.sub(r'\s+', ' ', original_text).strip()

                if not original_text:
                    continue

                if self.is_protected_content(original_text, len(block.get('lines', []))):
                    continue

                translated_text = self.translate_block(original_text)

                if translated_text.lower() != original_text.lower() and translated_text != "":
                    translated_blocks.append(TranslatedBlockV1(
                        block_id=block['block_id'],
                        original_text=original_text,
                        translated_text=translated_text
                    ))

            if translated_blocks:
                manifest_pages.append(PageTranslationPayload(
                    page_num=page['page_num'],
                    translated_blocks=translated_blocks
                ))

        manifest = TranslationManifest(
            file_name=file_name,
            pages=manifest_pages
        )

        os.makedirs(os.path.dirname(output_manifest_path), exist_ok=True)
        with open(output_manifest_path, 'w', encoding='utf-8') as f:
            manifest_dict = manifest.model_dump()
            f.write(json.dumps(manifest_dict, indent=2, ensure_ascii=False))

        print(f"\n[Translation Stage] Đã lưu Manifest tại: {output_manifest_path}")

# Lấy lại đường dẫn từ các cell trước
workspace_dir = "workspace"
document_v1_path = os.path.join(workspace_dir, "document.v1.json")
translation_manifest_path = os.path.join(workspace_dir, "translations", "translation-manifest.json")

# Khởi chạy luồng dịch thuật Local
translation_stage = TranslationStage()
translation_stage.run(document_v1_path, translation_manifest_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
[Translation Stage] Đang tải mô hình Qwen/Qwen2.5-7B-Instruct vào GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[Translation Stage] Mô hình đã sẵn sàng!
[Translation Stage] Bắt đầu dịch 11 trang bằng mô hình cục bộ...


Tiến trình dịch tài liệu:   0%|          | 0/11 [00:00<?, ?trang/s]


[Translation Stage] Đã lưu Manifest tại: workspace/translations/translation-manifest.json


In [14]:
import os
import json
import fitz
import urllib.request
from google.colab import files

class RenderStage:
    def __init__(self, font_dir="workspace/fonts"):
        self.font_dir = font_dir
        self.font_reg = os.path.join(font_dir, "Roboto-Regular.ttf")
        self.font_bold = os.path.join(font_dir, "Roboto-Bold.ttf")
        self._ensure_fonts_exist()

    def _ensure_fonts_exist(self):
        os.makedirs(self.font_dir, exist_ok=True)
        fonts = {
            self.font_reg: "https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Regular.ttf",
            self.font_bold: "https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Bold.ttf"
        }
        for path, url in fonts.items():
            if not os.path.exists(path):
                print(f"[Render Stage] Downloading font {os.path.basename(path)}...")
                urllib.request.urlretrieve(url, path)

    def get_optimal_fontsize(self, bbox: fitz.Rect, text_length: int, avg_original_fontsize: float) -> float:
        opt_fontsize = avg_original_fontsize
        min_fontsize = 6.0

        while opt_fontsize >= min_fontsize:
            char_width_ratio = opt_fontsize * 0.45
            chars_per_line = bbox.width / char_width_ratio
            if chars_per_line <= 0:
                break

            lines_needed = text_length / chars_per_line
            estimated_height = lines_needed * (opt_fontsize * 1.15)

            if estimated_height <= bbox.height:
                break

            opt_fontsize -= 0.25

        return max(opt_fontsize, min_fontsize)

    def is_block_bold(self, block) -> bool:
        for line in block.get('lines', []):
            for span in line.get('spans', []):
                font_name = span.get('font', '').lower()
                if 'bold' in font_name or 'black' in font_name:
                    return True
        return False

    def run(self, source_pdf_path: str, document_json_path: str, manifest_path: str, output_path: str):
        with open(document_json_path, 'r', encoding='utf-8') as f:
            document_data = json.load(f)

        with open(manifest_path, 'r', encoding='utf-8') as f:
            manifest_data = json.load(f)

        manifest_map = {}
        for p in manifest_data.get('pages', []):
            manifest_map[p['page_num']] = {b['block_id']: b['translated_text'] for b in p['translated_blocks']}

        # --- BƯỚC FIX LỖI XREF: THANH LỌC PDF TRƯỚC KHI XỬ LÝ ---
        raw_doc = fitz.open(source_pdf_path)
        # Sắp xếp lại file, xóa các object rác/hỏng và lưu vào bộ nhớ đệm (RAM)
        cleaned_pdf_bytes = raw_doc.tobytes(garbage=3, clean=True, deflate=True)
        raw_doc.close()

        # Mở lại file PDF từ bộ nhớ đệm đã được làm sạch
        doc = fitz.open("pdf", cleaned_pdf_bytes)
        # -------------------------------------------------------

        for page_num in range(len(doc)):
            if page_num not in manifest_map:
                continue

            page = doc[page_num]
            page.insert_font(fontname="ViRegular", fontfile=self.font_reg)
            page.insert_font(fontname="ViBold", fontfile=self.font_bold)

            doc_page = next((p for p in document_data['pages'] if p['page_num'] == page_num), None)
            if not doc_page:
                continue

            trans_map = manifest_map[page_num]

            for block in doc_page['blocks']:
                if block['block_id'] not in trans_map:
                    continue

                translated_text = trans_map[block['block_id']]
                is_bold = self.is_block_bold(block)

                total_size = 0
                span_count = 0

                for line in block['lines']:
                    for span in line['spans']:
                        span_bbox = fitz.Rect(span['bbox'])
                        span_bbox.x0 += 0.2
                        span_bbox.x1 -= 0.2
                        # Tắt chế độ cross-reference strict khi redact để tránh lỗi cảnh báo
                        page.add_redact_annot(span_bbox, fill=(1, 1, 1), cross_out=False)

                        total_size += span['size']
                        span_count += 1

                # Áp dụng redaction trên file đã được thanh lọc
                page.apply_redactions(images=fitz.PDF_REDACT_IMAGE_NONE)

                avg_fontsize = (total_size / span_count) if span_count > 0 else 10.0
                block_bbox = fitz.Rect(block['bbox'])

                text_bbox = fitz.Rect(block_bbox.x0 + 1, block_bbox.y0, block_bbox.x1 - 1, block_bbox.y1)
                opt_fontsize = self.get_optimal_fontsize(text_bbox, len(translated_text), avg_fontsize)

                selected_fontname = "ViBold" if is_bold else "ViRegular"
                selected_fontfile = self.font_bold if is_bold else self.font_reg

                page.insert_textbox(
                    text_bbox,
                    translated_text,
                    fontname=selected_fontname,
                    fontfile=selected_fontfile,
                    fontsize=opt_fontsize,
                    color=(0, 0, 0),
                    align=0
                )

        output_dir = os.path.dirname(output_path)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)

        doc.save(output_path, garbage=3, deflate=True)
        doc.close()
        print(f"[Render Stage] Đã lưu PDF với cấu trúc an toàn tuyệt đối: {output_path}")

# ==========================================
# KHỞI CHẠY GIAI ĐOẠN 3
# ==========================================
try:
    output_pdf_path = input_pdf.replace('.pdf', '_RetainPipeline_Vi.pdf')
except NameError:
    print("Vui lòng chạy lại Cell 1 để tải file PDF lên trước.")

render_stage = RenderStage()
render_stage.run(
    source_pdf_path=input_pdf,
    document_json_path=document_v1_path,
    manifest_path=translation_manifest_path,
    output_path=output_pdf_path
)

print(f"Hoàn tất quá trình! Đang tự động tải tệp xuống...")
files.download(output_pdf_path)

MuPDF error: format error: cannot find object in xref (260 0 R)

MuPDF error: format error: cannot find object in xref (265 0 R)

MuPDF error: format error: cannot find object in xref (306 0 R)

[Render Stage] Đã lưu PDF với cấu trúc an toàn tuyệt đối: 1-s2.0-S1877050925015984-main_RetainPipeline_Vi.pdf
Hoàn tất quá trình! Đang tự động tải tệp xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>